# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
import json
import os

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from typing import List, Optional, TypedDict, Union

from lib.state_machine import EntryPoint, Run, StateMachine, Step, Termination
from lib.tooling import Tool, ToolCall, tool
from lib.llm import LLM
from lib.messages import AIMessage, SystemMessage, ToolMessage, UserMessage
from lib.memory import ShortTermMemory
from lib.parsers import PydanticOutputParser

E0000 00:00:1779412507.842776 2131279 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1779412507.842814 2131279 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1779412507.842820 2131279 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1779412507.842825 2131279 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1779412507.842830 2131279 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


In [2]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [3]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    model_name="text-embedding-3-small"
)

chroma_client = chromadb.PersistentClient(path="chromadb")

collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)

@tool
def retrieve_game(query: str) -> list[dict]:
    """
    Semantic search: Finds most results in the vector DB

    Args:
        query: a question about game industry.

    You'll receive results as list. Each element contains:
        - Platform: like Game Boy, Playstation 5, Xbox 360...)
        - Name: Name of the Game
        - YearOfRelease: Year when that game was released for that platform
        - Description: Additional details about the game
    """
    results = collection.query(
        query_texts=[query],
        n_results=5,
    )

    games = []
    
    for metadata in results.get("metadatas", [[]])[0]:
        games.append({
            "Platform": metadata.get("Platform"),
            "Name": metadata.get("Name"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description"),
        })

    return games

#### Evaluate Retrieval Tool

In [4]:
class EvaluationReport(BaseModel):
    """Evaluation of whether retrieved documents can answer a question."""

    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")

@tool
def evaluate_retrieval(question: str, retrieved_docs: list[dict]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    Args:
        question: original question from user
        retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
        - useful: whether the documents are useful to answer the question
        - description: description about the evaluation result
    """
    json_docs = json.dumps(retrieved_docs, indent=2, ensure_ascii=False)

    prompt = f"""
        Your task is to evaluate if the documents are enough to respond the query.
        Give a detailed explanation, so it's possible to take an action to accept it or not.

        Question:
        {question}

        Retrieved documents:
        {json_docs}

        Return useful=true only when the retrieved documents contain enough relevant,
        specific information to answer the user's question accurately. Return useful=false
        when the documents are missing the answer, are about different games/platforms,
        or require external knowledge to answer confidently.
        """

    judge = LLM(model="gpt-4o-mini", temperature=0)
    
    judge_response = judge.invoke(
        input=prompt,
        response_format=EvaluationReport,
    )

    parser = PydanticOutputParser(model_class=EvaluationReport)
    report = parser.parse(judge_response)
    return report.model_dump()

#### Game Web Search Tool

In [5]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(question: str) -> list[dict]:
    """
    Semantic search: Finds most results in the vector DB

    Args:
        question: a question about game industry.

    You'll receive results as list. Each element contains:
        - title: Title of the web page
        - url: URL of the web page
        - content: Search result content related to the question
        - score: Tavily relevance score when available
    """
    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5,
    )

    return [
        {
            "title": result.get("title"),
            "url": result.get("url"),
            "content": result.get("content"),
            "score": result.get("score"),
        }
        for result in response.get("results", [])
    ]

### Agent

In [6]:
class AgentState(TypedDict):
    user_query: str  # The current user query being processed
    instructions: str  # System instructions for the agent
    messages: List[dict]  # List of conversation messages
    current_tool_calls: Optional[List[ToolCall]]  # Current pending tool calls
    session_id: str  # Conversation session identifier


class Agent:
    def __init__(self, 
                 model_name: str,
                 instructions: str, 
                 tools: List[Tool] = None,
                 temperature: float = 0.7):
        """
        Initialize an Agent instance
        
        Args:
            model_name: Name/identifier of the LLM model to use
            instructions: System instructions for the agent
            tools: Optional list of tools available to the agent
            temperature: Temperature parameter for LLM (default: 0.7)
        """
        self.instructions = instructions
        self.tools = tools if tools else []
        self.model_name = model_name
        self.temperature = temperature
                
        # Initialize memory and state machine
        self.memory = ShortTermMemory()
        self.workflow = self._create_state_machine()

    def _prepare_messages_step(self, state: AgentState) -> AgentState:
        """Step logic: Prepare messages for LLM consumption"""

        messages = state.get("messages", [])

        if not messages:
            messages = [SystemMessage(content=state["instructions"])]

        messages.append(UserMessage(content=state["user_query"]))
        
        return {
            "messages": messages,
            "session_id": state["session_id"],
        }

    def _llm_step(self, state: AgentState) -> AgentState:
        """Step logic: Process the current state through the LLM"""

        # Initialize LLM
        llm = LLM(
            model=self.model_name,
            temperature=self.temperature,
            tools=self.tools
        )

        response = llm.invoke(state["messages"])
        tool_calls = response.tool_calls if response.tool_calls else None

        # Create AI message with content and tool calls
        ai_message = AIMessage(content=response.content, tool_calls=tool_calls)
        
        return {
            "messages": state["messages"] + [ai_message],
            "current_tool_calls": tool_calls,
            "session_id": state["session_id"],
        }

    def _tool_step(self, state: AgentState) -> AgentState:
        """Step logic: Execute any pending tool calls"""
        tool_calls = state["current_tool_calls"] or []
        tool_messages = []
        
        for call in tool_calls:
            # Access tool call data correctly
            function_name = call.function.name
            function_args = json.loads(call.function.arguments)
            tool_call_id = call.id
            # Find the matching tool
            tool = next((t for t in self.tools if t.name == function_name), None)
            if tool:
                result = tool(**function_args)
                tool_messages.append(
                    ToolMessage(
                        content=json.dumps(result), 
                        tool_call_id=tool_call_id, 
                        name=function_name, 
                    )
                )

        # Clear tool calls and add results to messages
        return {
            "messages": state["messages"] + tool_messages,
            "current_tool_calls": None,
            "session_id": state["session_id"],
        }

    def _create_state_machine(self) -> StateMachine[AgentState]:
        """Create the internal state machine for the agent"""
        machine = StateMachine[AgentState](AgentState)
        
        # Create steps
        entry = EntryPoint[AgentState]()
        message_prep = Step[AgentState]("message_prep", self._prepare_messages_step)
        llm_processor = Step[AgentState]("llm_processor", self._llm_step)
        tool_executor = Step[AgentState]("tool_executor", self._tool_step)
        termination = Termination[AgentState]()
        
        machine.add_steps([entry, message_prep, llm_processor, tool_executor, termination])
        
        # Add transitions
        machine.connect(entry, message_prep)
        machine.connect(message_prep, llm_processor)
        
        # Transition based on whether there are tool calls
        def check_tool_calls(state: AgentState) -> Union[Step[AgentState], str]:
            """Transition logic: Check if there are tool calls"""
            if state.get("current_tool_calls"):
                return tool_executor
            return termination

        machine.connect(llm_processor, [tool_executor, termination], check_tool_calls)
        machine.connect(tool_executor, llm_processor)  # Go back to llm after tool execution
        
        return machine

    def invoke(self, query: str, session_id: Optional[str] = None) -> Run:
        """
        Run the agent on a query
        
        Args:
            query: The user's query to process
            session_id: Optional conversation session identifier (uses "default" if None)
            
        Returns:
            The final run object after processing
        """

        session_id = session_id or "default"
        self.memory.create_session(session_id)

        previous_messages = []
        last_run = self.memory.get_last_object(session_id)
        if last_run:
            last_state = last_run.get_final_state()
            if last_state:
                previous_messages = last_state["messages"]

        initial_state: AgentState = {
            "user_query": query,
            "instructions": self.instructions,
            "messages": previous_messages,
            "current_tool_calls": None,
            "session_id": session_id,
        }

        run_object = self.workflow.run(initial_state)
        self.memory.add(run_object, session_id)

        return run_object

    def get_session_runs(self, session_id: Optional[str] = None) -> List[Run]:
        """Return all completed runs for a session."""
        return self.memory.get_all_objects(session_id)

    def reset_session(self, session_id: Optional[str] = None):
        """Reset memory for a session."""
        self.memory.reset(session_id)


udaplay_agent_instructions = """
You are a careful AI research agent for video game industry questions.

Use the available tools:
1. Start with retrieve_game for questions about games, platforms, release years, descriptions, or availability.
2. After retrieve_game returns documents, call evaluate_retrieval with the original user question and the retrieved documents.
3. If evaluate_retrieval says the documents are useful, answer from the retrieved documents.
4. If evaluate_retrieval says the documents are not useful, call game_web_search and answer from the web results.

When answering, be concise and factual. Always report whether the answer comes from internal game data versus web search.

Format every final answer as:

Answer:
Concise answer

Evidence:
Key supporting fact from retrieved game data or web result

Source:
Internal UdaPlay game database: game name, platform, release year, publisher
Web: url, game name, platform, release year, publisher

Only include source types that were actually used. If web search results are used, include the source URLs.
If the tools do not provide enough evidence, say what is missing instead of guessing.
"""

udaplay_tools = [
    retrieve_game,
    evaluate_retrieval,
    game_web_search,
]

udaplay_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=udaplay_agent_instructions,
    tools=udaplay_tools,
    temperature=0,
)

### Invoke Agent

In [7]:
questions = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for i in range(len(questions)):
    udaplay_agent.reset_session()
    run_object = udaplay_agent.invoke(questions[i], session_id=f"Udaplay {i}")
    final_state = run_object.get_final_state()
    messages = final_state["messages"]
    answer = messages[-1].content
    tools_used = [message.name for message in messages if message.role == "tool"]

    print(f"\nQuestion: {questions[i]}")
    print("\nTool usage:")
    for tool_name in tools_used:
        print(f"- {tool_name}")
    print(f"\n{answer}\n")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Question: When was Pokémon Gold and Silver released?

Tool usage:
- retrieve_game
- evaluate_retrieval

Answer:
Pokémon Gold and Silver were released in 1999.

Evidence:
The games were released for the Game Boy Color in 1999.

Source:
Internal UdaPlay game database: Pokémon Gold and Silver, Game Boy Color, 1999, Nintendo.

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Termina